In [1]:
from datasets import load_dataset
from transformers import PLBartTokenizer, PLBartForConditionalGeneration, AdamW
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm

dataset = load_dataset('code_x_glue_ct_code_to_text', 'python')

small_train_dataset = dataset['train'].select(range(int(len(dataset['train']) * 0.01)))
small_valid_dataset = dataset['validation'].select(range(int(len(dataset['validation']) * 0.01)))
small_test_dataset = dataset['test'].select(range(int(len(dataset['test']) * 0.01)))

print(f"Reduced training set size: {len(small_train_dataset)}")
print(f"Reduced validation set size: {len(small_valid_dataset)}")
print(f"Reduced test set size: {len(small_test_dataset)}")

README.md:   0%|          | 0.00/26.7k [00:00<?, ?B/s]

train-00000-of-00002.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/147M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/18.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/251820 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13914 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/14918 [00:00<?, ? examples/s]

Reduced training set size: 2518
Reduced validation set size: 139
Reduced test set size: 149


In [2]:
tokenizer = PLBartTokenizer.from_pretrained("uclanlp/plbart-python-en_XX", src_lang="python", tgt_lang="en_XX")


def tokenize_function(examples):
    code_tokens = tokenizer(
        examples['code'],
        padding="max_length",
        truncation=True,
        max_length=384,  
        return_tensors="pt"
    )
    docstring_tokens = tokenizer(
        examples['docstring'],
        padding="max_length",
        truncation=True,
        max_length=128,  
        return_tensors="pt"
    )
    return {
        'input_ids': code_tokens['input_ids'],
        'attention_mask': code_tokens['attention_mask'],
        'labels': docstring_tokens['input_ids']
    }


tokenized_train_dataset = small_train_dataset.map(tokenize_function, batched=True)
tokenized_valid_dataset = small_valid_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = small_test_dataset.map(tokenize_function, batched=True)

sentencepiece.bpe.model:   0%|          | 0.00/986k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/783 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/2518 [00:00<?, ? examples/s]

Map:   0%|          | 0/139 [00:00<?, ? examples/s]

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

In [3]:

def collate_fn(batch):
    input_ids = torch.stack([torch.tensor(f['input_ids']) for f in batch])
    attention_mask = torch.stack([torch.tensor(f['attention_mask']) for f in batch])
    
    labels = [" ".join(f['docstring_tokens']) for f in batch]
    
    tokenized_labels = tokenizer(labels, padding=True, truncation=True, return_tensors="pt")
    
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': tokenized_labels['input_ids']
    }

train_loader = DataLoader(tokenized_train_dataset, batch_size=24, collate_fn=collate_fn)
valid_loader = DataLoader(tokenized_valid_dataset, batch_size=24, collate_fn=collate_fn)
test_loader = DataLoader(tokenized_test_dataset, batch_size=24, collate_fn=collate_fn)

In [4]:
model = PLBartForConditionalGeneration.from_pretrained("uclanlp/plbart-python-en_XX")

optimizer = AdamW(model.parameters(), lr=5e-5)

pytorch_model.bin:   0%|          | 0.00/557M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [5]:
from tqdm import tqdm
import torch
from collections import deque
from statistics import mean

def evaluate_model(model, valid_loader, device="cuda"):
    model.to(device)
    model.eval()  
    
    total_loss = 0
    num_batches = len(valid_loader)
    
    with torch.no_grad():  
        for batch in valid_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
    
    avg_loss = total_loss / num_batches
    return avg_loss

def train_model(model, train_loader, valid_loader, optimizer, device="cuda", num_epochs=3):
    model.to(device)
    model.train()
    
    loss_window = deque(maxlen=100) 
    total_batches = len(train_loader) * num_epochs
    
    progress_bar = tqdm(
        total=total_batches,
        desc='Training Progress',
        ncols=100,
        dynamic_ncols=False,
        position=0,
        leave=True
    )
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        model.train() 
        
        for batch_idx, batch in enumerate(train_loader):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            
            current_loss = loss.item()
            loss_window.append(current_loss)
            epoch_loss += current_loss
            
            running_avg_loss = mean(loss_window)
            epoch_avg_loss = epoch_loss / (batch_idx + 1)
            
            progress_bar.set_description(f'Epoch {epoch + 1}/{num_epochs}')
            progress_bar.set_postfix({
                'loss': f'{current_loss:.4f}',
                'avg': f'{running_avg_loss:.4f}',
                'epoch_avg': f'{epoch_avg_loss:.4f}'
            })
            progress_bar.update(1)
        
        valid_loss = evaluate_model(model, valid_loader, device)
        print(f"\nEpoch {epoch + 1} Validation Loss: {valid_loss:.4f}")
        
        torch.cuda.empty_cache()
    
    progress_bar.close()

In [6]:
train_model(model, train_loader,valid_loader, optimizer,num_epochs=10)

Epoch 1/10:  10%|▌    | 105/1050 [01:05<09:15,  1.70it/s, loss=0.1060, avg=2.0288, epoch_avg=2.2573]


Epoch 1 Validation Loss: 0.1484


Epoch 2/10:  20%|█    | 210/1050 [02:11<08:17,  1.69it/s, loss=0.0603, avg=0.1084, epoch_avg=0.1091]


Epoch 2 Validation Loss: 0.1388


Epoch 3/10:  30%|█▌   | 315/1050 [03:17<07:15,  1.69it/s, loss=0.0516, avg=0.0858, epoch_avg=0.0850]


Epoch 3 Validation Loss: 0.1293


... (10 outputs left out)

Epoch 9/10:  90%|████▌| 945/1050 [09:57<01:02,  1.68it/s, loss=0.0082, avg=0.0326, epoch_avg=0.0320]


Epoch 9 Validation Loss: 0.1363


Epoch 10/10: 100%|███| 1050/1050 [11:05<00:00,  1.58it/s, loss=0.0054, avg=0.0290, epoch_avg=0.0285]


Epoch 10 Validation Loss: 0.1617


In [7]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu
from tqdm import tqdm
import torch

def generate_predictions(model, dataloader, tokenizer, device="cuda"):
    model.to(device)
    model.eval()
    predictions = []
    references = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_start_token_id=tokenizer.lang_code_to_id["__en_XX__"],
                max_length=150,
                num_beams=4,
                early_stopping=True
            )
            
            preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            refs = tokenizer.batch_decode(batch['labels'], skip_special_tokens=True)

            for i in range(3):  
                print(f"Generated: '{preds[i]}'")
                print(f"Reference: '{refs[i]}'\n")
            
            predictions.extend(preds)
            references.extend([[ref.split()] for ref in refs])  
    
    return predictions, references


def compute_bleu(predictions, references):
    tokenized_predictions = [pred.split() for pred in predictions]
    
    bleu_score = corpus_bleu(references, tokenized_predictions)
    return bleu_score

val_predictions, val_references = generate_predictions(model, valid_loader, tokenizer)
val_bleu_score = compute_bleu(val_predictions, val_references)
print(f"Validation BLEU Score: {val_bleu_score:.2f}")

test_predictions, test_references = generate_predictions(model, test_loader, tokenizer)
test_bleu_score = compute_bleu(test_predictions, test_references)
print(f"Test BLEU Score: {test_bleu_score:.2f}")


 17%|█▋        | 1/6 [00:01<00:05,  1.15s/it]

Generated: 'Save model to a pickle located at path'
Reference: 'Save model to a pickle located at path'

Generated: 'Acquire from Nature paper.'
Reference: 'CNN from Nature paper.'

Generated: 'convolutions - only net'
Reference: 'convolutions - only net'



 33%|███▎      | 2/6 [00:01<00:03,  1.07it/s]

Generated: 'Update the estimate.'
Reference: 'Update the estimate.'

Generated: 'Decorator provided method args as instance attributes.'
Reference: 'Stores provided method args as instance attributes.'

Generated: 'Flattens a variables and their gradients.'
Reference: 'Flattens a variables and their gradients.'



 50%|█████     | 3/6 [00:03<00:03,  1.04s/it]

Generated: 'Draw the keypoint onto a given image. The keypoint is drawn as a square.'
Reference: 'Draw the keypoint onto a given image.'

Generated: 'Generate nearby points to this keypoint based on manhattan distance.'
Reference: 'Generate nearby points to this keypoint based on manhattan distance.'

Generated: 'Create a shallow copy of the Keypoint object.'
Reference: 'Create a shallow copy of the Keypoint object.'



... (14 outputs left out)

 71%|███████▏  | 5/7 [00:05<00:02,  1.15s/it]

Generated: 'Returns the Cloud Function with the given name.'
Reference: 'Returns the Cloud Function with the given name.'

Generated: 'Creates a new function in Cloud Function in the location specified in the body.'
Reference: 'Creates a new function in Cloud Function in the location specified in the body.'

Generated: 'Updates Cloud according to the specified update mask.'
Reference: 'Updates Cloud Functions according to the specified update mask.'



100%|██████████| 7/7 [00:07<00:00,  1.03s/it]

Generated: 'Print out stats about how files are getting processed.'
Reference: 'Print out stats about how files are getting processed.'

Generated: 'Update this with a new set of paths to DAG definition files.'
Reference: 'Update this with a new set of paths to DAG definition files.'

Generated: 'sleeps until all the processors are done.'
Reference: 'Sleeps until all the processors are done.'

Generated: 'Roll back a transaction.'
Reference: 'Roll back a transaction.'

Generated: 'Run a query for entities.'
Reference: 'Run a query for entities.'

Generated: 'Gets the latest state of a long - running operation.'
Reference: 'Gets the latest state of a long - running operation.'

Test BLEU Score: 0.80


In [8]:
pip install rouge-score

/usr/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=79b2f487b93482160e861e0e14c282577d6193789ce044b95f1e794baf858fac
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score
Note: you may need to restart the kernel to use updated packages.


In [9]:
from rouge_score import rouge_scorer, scoring

rouge_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_aggregator = scoring.BootstrapAggregator()

def compute_rouge(predictions, references):
    for pred, ref in zip(predictions, references):
        if isinstance(ref, list):
            ref = ' '.join(ref) if all(isinstance(item, str) for item in ref) else ' '.join(map(str, ref))
        if isinstance(pred, list):
            pred = ' '.join(pred) if all(isinstance(item, str) for item in pred) else ' '.join(map(str, pred))
        
        scores = rouge_scorer.score(ref, pred)
        rouge_aggregator.add_scores(scores)
    
    result = rouge_aggregator.aggregate()
    return result


In [10]:
val_predictions, val_references = generate_predictions(model, valid_loader, tokenizer)
val_bleu_score = compute_bleu(val_predictions, val_references)
val_rouge_score = compute_rouge(val_predictions, val_references)

print(f"Validation BLEU Score: {val_bleu_score:.2f}")
print("Validation ROUGE Scores:")
print(f"  ROUGE-1: {val_rouge_score['rouge1'].mid.fmeasure:.2f}")
print(f"  ROUGE-2: {val_rouge_score['rouge2'].mid.fmeasure:.2f}")
print(f"  ROUGE-L: {val_rouge_score['rougeL'].mid.fmeasure:.2f}")

test_predictions, test_references = generate_predictions(model, test_loader, tokenizer)
test_bleu_score = compute_bleu(test_predictions, test_references)
test_rouge_score = compute_rouge(test_predictions, test_references)

print(f"Test BLEU Score: {test_bleu_score:.2f}")
print("Test ROUGE Scores:")
print(f"  ROUGE-1: {test_rouge_score['rouge1'].mid.fmeasure:.2f}")
print(f"  ROUGE-2: {test_rouge_score['rouge2'].mid.fmeasure:.2f}")
print(f"  ROUGE-L: {test_rouge_score['rougeL'].mid.fmeasure:.2f}")

 17%|█▋        | 1/6 [00:00<00:04,  1.21it/s]

Generated: 'Save model to a pickle located at path'
Reference: 'Save model to a pickle located at path'

Generated: 'Acquire from Nature paper.'
Reference: 'CNN from Nature paper.'

Generated: 'convolutions - only net'
Reference: 'convolutions - only net'



 33%|███▎      | 2/6 [00:01<00:03,  1.26it/s]

Generated: 'Update the estimate.'
Reference: 'Update the estimate.'

Generated: 'Decorator provided method args as instance attributes.'
Reference: 'Stores provided method args as instance attributes.'

Generated: 'Flattens a variables and their gradients.'
Reference: 'Flattens a variables and their gradients.'



 50%|█████     | 3/6 [00:02<00:02,  1.03it/s]

Generated: 'Draw the keypoint onto a given image. The keypoint is drawn as a square.'
Reference: 'Draw the keypoint onto a given image.'

Generated: 'Generate nearby points to this keypoint based on manhattan distance.'
Reference: 'Generate nearby points to this keypoint based on manhattan distance.'

Generated: 'Create a shallow copy of the Keypoint object.'
Reference: 'Create a shallow copy of the Keypoint object.'



... (16 outputs left out)

Generated: 'Returns the Cloud Function with the given name.'
Reference: 'Returns the Cloud Function with the given name.'

Generated: 'Creates a new function in Cloud Function in the location specified in the body.'
Reference: 'Creates a new function in Cloud Function in the location specified in the body.'

Generated: 'Updates Cloud according to the specified update mask.'
Reference: 'Updates Cloud Functions according to the specified update mask.'



100%|██████████| 7/7 [00:07<00:00,  1.04s/it]

Generated: 'Print out stats about how files are getting processed.'
Reference: 'Print out stats about how files are getting processed.'

Generated: 'Update this with a new set of paths to DAG definition files.'
Reference: 'Update this with a new set of paths to DAG definition files.'

Generated: 'sleeps until all the processors are done.'
Reference: 'Sleeps until all the processors are done.'

Generated: 'Roll back a transaction.'
Reference: 'Roll back a transaction.'

Generated: 'Run a query for entities.'
Reference: 'Run a query for entities.'

Generated: 'Gets the latest state of a long - running operation.'
Reference: 'Gets the latest state of a long - running operation.'



Test BLEU Score: 0.80
Test ROUGE Scores:
  ROUGE-1: 0.90
  ROUGE-2: 0.88
  ROUGE-L: 0.90


**Code T5 Model**

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-base")

tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [3]:
def tokenize_function(examples):
    code_tokens = tokenizer(
        examples['code'],
        padding="max_length",
        truncation=True,
        max_length=384,
        return_tensors="pt"
    )
    docstring_tokens = tokenizer(
        examples['docstring'],
        padding="max_length",
        truncation=True,
        max_length=128,  
        return_tensors="pt"
    )
    return {
        'input_ids': code_tokens['input_ids'],
        'attention_mask': code_tokens['attention_mask'],
        'labels': docstring_tokens['input_ids']
    }


tokenized_train_dataset = small_train_dataset.map(tokenize_function, batched=True)
tokenized_valid_dataset = small_valid_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = small_test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/2518 [00:00<?, ? examples/s]

Map:   0%|          | 0/139 [00:00<?, ? examples/s]

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

In [4]:
def collate_fn(batch):
    input_ids = torch.stack([torch.tensor(f['input_ids']) for f in batch])
    attention_mask = torch.stack([torch.tensor(f['attention_mask']) for f in batch])
    
    labels = [" ".join(f['docstring_tokens']) for f in batch]
    
    tokenized_labels = tokenizer(labels, padding=True, truncation=True, return_tensors="pt")
    
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': tokenized_labels['input_ids']
    }

train_loader = DataLoader(tokenized_train_dataset, batch_size=8, collate_fn=collate_fn)
valid_loader = DataLoader(tokenized_valid_dataset, batch_size=8, collate_fn=collate_fn)
test_loader = DataLoader(tokenized_test_dataset, batch_size=8, collate_fn=collate_fn)

In [5]:
optimizer = AdamW(model.parameters(), lr=5e-5)

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [7]:
train_model(model, train_loader,valid_loader, optimizer,num_epochs=10)

Epoch 1/10:  10%|▌    | 315/3150 [02:18<19:03,  2.48it/s, loss=0.0093, avg=0.0413, epoch_avg=0.2080]


Epoch 1 Validation Loss: 0.0188


Epoch 2/10:  20%|█    | 630/3150 [04:39<16:56,  2.48it/s, loss=0.0011, avg=0.0118, epoch_avg=0.0108]


Epoch 2 Validation Loss: 0.0085


Epoch 3/10:  30%|█▌   | 945/3150 [07:00<14:48,  2.48it/s, loss=0.0005, avg=0.0047, epoch_avg=0.0071]


Epoch 3 Validation Loss: 0.0150


... (10 outputs left out)

Epoch 9/10:  90%|███▌| 2835/3150 [21:06<02:07,  2.48it/s, loss=0.0003, avg=0.0050, epoch_avg=0.0033]


Epoch 9 Validation Loss: 0.0133


Epoch 10/10: 100%|███| 3150/3150 [23:30<00:00,  2.23it/s, loss=0.0010, avg=0.0498, epoch_avg=0.0184]


Epoch 10 Validation Loss: 0.0205


In [10]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu
from tqdm import tqdm
import torch

def generate_predictions(model, dataloader, tokenizer, device="cuda"):
    model.to(device)
    model.eval()
    predictions = []
    references = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=150,
                num_beams=4,
                early_stopping=True
            )
            
            preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            refs = tokenizer.batch_decode(batch['labels'], skip_special_tokens=True)
            
            for i in range(3): 
                print(f"Generated: '{preds[i]}'")
                print(f"Reference: '{refs[i]}'\n")
            
            predictions.extend(preds)
            references.extend([[ref.split()] for ref in refs])  
    
    return predictions, references


def compute_bleu(predictions, references):
    tokenized_predictions = [pred.split() for pred in predictions]
    
    bleu_score = corpus_bleu(references, tokenized_predictions)
    return bleu_score

val_predictions, val_references = generate_predictions(model, valid_loader, tokenizer)
val_bleu_score = compute_bleu(val_predictions, val_references)
print(f"Validation BLEU Score: {val_bleu_score:.2f}")

test_predictions, test_references = generate_predictions(model, test_loader, tokenizer)
test_bleu_score = compute_bleu(test_predictions, test_references)
print(f"Test BLEU Score: {test_bleu_score:.2f}")

  6%|▌         | 1/18 [00:02<00:38,  2.28s/it]

Generated: 'Save model to a pickle located at path'
Reference: 'Save model to a pickle located at path'

Generated: 'CNN from Nature paper.'
Reference: 'CNN from Nature paper.'

Generated: 'convolutions - only net'
Reference: 'convolutions - only net'



 11%|█         | 2/18 [00:03<00:23,  1.46s/it]

Generated: 'Create placeholder to feed observations into of the size appropriate to the observation space and add input encoder of the appropriate type.'
Reference: 'Create placeholder to feed observations into of the size appropriate to the observation space and add input encoder of the appropriate type.'

Generated: 'Encode input in the way that is appropriate to the observation space'
Reference: 'Encode input in the way that is appropriate to the observation space'

Generated: 'Pickles the current policy for later inspection.'
Reference: 'Pickles the current policy for later inspection.'



 17%|█▋        | 3/18 [00:04<00:18,  1.23s/it]

Generated: 'See Schedule. value'
Reference: 'See Schedule. value'

Generated: 'Control a single environment instance using IPC and shared memory.'
Reference: 'Control a single environment instance using IPC and shared memory.'

Generated: 'Main entrypoint for A2C algorithm. Train a policy with given network architecture on a given environment using a2c algorithm.'
Reference: 'Main entrypoint for A2C algorithm. Train a policy with given network architecture on a given environment using a2c algorithm.'



... (64 outputs left out)

 95%|█████████▍| 18/19 [00:18<00:01,  1.18s/it]

Generated: 'Returns the number of slots open at the moment'
Reference: 'Returns the number of slots open at the moment'

Generated: 'Runs command and returns stdout'
Reference: 'Runs command and returns stdout'

Generated: 'Remove an option if it exists in config from a file or default config. If both of config have the same option this removes the option in both configs unless remove_default = False.'
Reference: 'Remove an option if it exists in config from a file or default config. If both of config have the same option this removes the option in both configs unless remove_default = False.'



100%|██████████| 19/19 [00:18<00:00,  1.03it/s]

Generated: 'Roll back a transaction.'
Reference: 'Roll back a transaction.'

Generated: 'Run a query for entities.'
Reference: 'Run a query for entities.'

Generated: 'Gets the latest state of a long - running operation.'
Reference: 'Gets the latest state of a long - running operation.'

Test BLEU Score: 0.94


In [13]:
val_predictions, val_references = generate_predictions(model, valid_loader, tokenizer)
val_bleu_score = compute_bleu(val_predictions, val_references)
val_rouge_score = compute_rouge(val_predictions, val_references)

print(f"Validation BLEU Score: {val_bleu_score:.2f}")
print("Validation ROUGE Scores:")
print(f"  ROUGE-1: {val_rouge_score['rouge1'].mid.fmeasure:.2f}")
print(f"  ROUGE-2: {val_rouge_score['rouge2'].mid.fmeasure:.2f}")
print(f"  ROUGE-L: {val_rouge_score['rougeL'].mid.fmeasure:.2f}")

test_predictions, test_references = generate_predictions(model, test_loader, tokenizer)
test_bleu_score = compute_bleu(test_predictions, test_references)
test_rouge_score = compute_rouge(test_predictions, test_references)

print(f"Test BLEU Score: {test_bleu_score:.2f}")
print("Test ROUGE Scores:")
print(f"  ROUGE-1: {test_rouge_score['rouge1'].mid.fmeasure:.2f}")
print(f"  ROUGE-2: {test_rouge_score['rouge2'].mid.fmeasure:.2f}")
print(f"  ROUGE-L: {test_rouge_score['rougeL'].mid.fmeasure:.2f}")

  6%|▌         | 1/18 [00:02<00:34,  2.04s/it]

Generated: 'Save model to a pickle located at path'
Reference: 'Save model to a pickle located at path'

Generated: 'CNN from Nature paper.'
Reference: 'CNN from Nature paper.'

Generated: 'convolutions - only net'
Reference: 'convolutions - only net'



 11%|█         | 2/18 [00:02<00:21,  1.37s/it]

Generated: 'Create placeholder to feed observations into of the size appropriate to the observation space and add input encoder of the appropriate type.'
Reference: 'Create placeholder to feed observations into of the size appropriate to the observation space and add input encoder of the appropriate type.'

Generated: 'Encode input in the way that is appropriate to the observation space'
Reference: 'Encode input in the way that is appropriate to the observation space'

Generated: 'Pickles the current policy for later inspection.'
Reference: 'Pickles the current policy for later inspection.'



 17%|█▋        | 3/18 [00:03<00:17,  1.18s/it]

Generated: 'See Schedule. value'
Reference: 'See Schedule. value'

Generated: 'Control a single environment instance using IPC and shared memory.'
Reference: 'Control a single environment instance using IPC and shared memory.'

Generated: 'Main entrypoint for A2C algorithm. Train a policy with given network architecture on a given environment using a2c algorithm.'
Reference: 'Main entrypoint for A2C algorithm. Train a policy with given network architecture on a given environment using a2c algorithm.'



... (65 outputs left out)

Generated: 'Returns the number of slots open at the moment'
Reference: 'Returns the number of slots open at the moment'

Generated: 'Runs command and returns stdout'
Reference: 'Runs command and returns stdout'

Generated: 'Remove an option if it exists in config from a file or default config. If both of config have the same option this removes the option in both configs unless remove_default = False.'
Reference: 'Remove an option if it exists in config from a file or default config. If both of config have the same option this removes the option in both configs unless remove_default = False.'



100%|██████████| 19/19 [00:18<00:00,  1.02it/s]

Generated: 'Roll back a transaction.'
Reference: 'Roll back a transaction.'

Generated: 'Run a query for entities.'
Reference: 'Run a query for entities.'

Generated: 'Gets the latest state of a long - running operation.'
Reference: 'Gets the latest state of a long - running operation.'



Test BLEU Score: 0.94
Test ROUGE Scores:
  ROUGE-1: 0.98
  ROUGE-2: 0.98
  ROUGE-L: 0.98
